# 22 - Native Ternary MLP Runtime Prototype

Protótipo estrutural do runtime nativo para o caminho `packed recovered L78`.

Objetivo desta fase:
- substituir os 78 MLPs por `PackedTernaryLinear`;
- manter os bytes packed como buffer persistente;
- preservar o adapter recovered via PEFT;
- testar geração e, opcionalmente, as 100 perguntas;
- medir carga, VRAM/RAM e throughput contra o notebook 20.

Importante: este é `native_structural_prototype`, não kernel otimizado. Ele desempacota no `forward`, então pode ser mais lento. A meta aqui é provar compatibilidade funcional antes de CUDA/Triton.

In [ ]:
# Dependências principais. Rode se o ambiente ainda não tiver os pacotes.
import sys, subprocess, importlib.util

required = {
    'torch': 'torch',
    'transformers': 'transformers',
    'accelerate': 'accelerate',
    'peft': 'peft',
    'safetensors': 'safetensors',
    'sentencepiece': 'sentencepiece',
    'psutil': 'psutil',
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Instalando dependências ausentes:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('Dependências principais já disponíveis.')

# O adapter recovered foi salvo com campos PEFT novos. Atualizar evita warnings como
# Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'].
UPGRADE_PEFT = True
if UPGRADE_PEFT:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'peft'])

In [ ]:
# Bootstrap Colab/local e paths oficiais.
from pathlib import Path
import os, sys, json, time, gc, shutil, platform

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)

CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
EXPORTS_DIR = PROJECT_ROOT / 'exports'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'packed_recovery'
RUNTIME_DIR = PROJECT_ROOT / 'runtime' / 'packed_recovered_l78'
DIST_DIR = PROJECT_ROOT / 'build' / 'distributable_packed_recovered_l78'
DATA_PATH = PROJECT_ROOT / 'data' / 'gemma4_100_questions.json'

for path in [RESULTS_DIR, DIST_DIR]:
    path.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(RUNTIME_DIR))
print('PROJECT_ROOT =', PROJECT_ROOT)
print('RUNTIME_DIR =', RUNTIME_DIR)
print('DIST_DIR =', DIST_DIR)

In [ ]:
# Garante que a pasta distribuível exista. Se já existir, apenas valida o layout.
STUDENT_DIR = CHECKPOINTS_DIR / 'student_pruned'
ADAPTER_DIR = CHECKPOINTS_DIR / 'student_packed_ternary_recovered_l78_t4_resume'
PACKED_SOURCE_DIR = next((p for p in [EXPORTS_DIR / 'ternary_mlp_l78', DEPLOY_DIR] if (p / 'manifest.json').exists() and (p / 'mlp_ternary_2bit.bin').exists()), None)

def copytree_clean(src, dst):
    src = Path(src)
    dst = Path(dst)
    if dst.exists() or dst.is_symlink():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            shutil.rmtree(dst)
    shutil.copytree(src, dst)

def copy_file_clean(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    shutil.copy2(src, dst)

def ensure_distributable():
    required = [
        DIST_DIR / 'student_pruned',
        DIST_DIR / 'adapter_recovered_l78',
        DIST_DIR / 'packed' / 'manifest.json',
        DIST_DIR / 'packed' / 'mlp_ternary_2bit.bin',
        DIST_DIR / 'runtime' / 'native_loader.py',
        DIST_DIR / 'runtime' / 'run_native_inference.py',
    ]
    if all(p.exists() for p in required):
        return 'existing'
    if PACKED_SOURCE_DIR is None:
        raise FileNotFoundError('Packed source not found in exports/ternary_mlp_l78 or deploy')
    missing_sources = [str(p) for p in [STUDENT_DIR, ADAPTER_DIR, RUNTIME_DIR, PACKED_SOURCE_DIR] if not Path(p).exists()]
    if missing_sources:
        raise FileNotFoundError('Missing sources:\n' + '\n'.join(missing_sources))
    if DIST_DIR.exists():
        shutil.rmtree(DIST_DIR)
    DIST_DIR.mkdir(parents=True, exist_ok=True)
    copytree_clean(STUDENT_DIR, DIST_DIR / 'student_pruned')
    copytree_clean(ADAPTER_DIR, DIST_DIR / 'adapter_recovered_l78')
    copytree_clean(RUNTIME_DIR, DIST_DIR / 'runtime')
    (DIST_DIR / 'packed').mkdir(parents=True, exist_ok=True)
    copy_file_clean(PACKED_SOURCE_DIR / 'manifest.json', DIST_DIR / 'packed' / 'manifest.json')
    copy_file_clean(PACKED_SOURCE_DIR / 'mlp_ternary_2bit.bin', DIST_DIR / 'packed' / 'mlp_ternary_2bit.bin')
    return 'created'

dist_status = ensure_distributable()
print('distributable:', dist_status, DIST_DIR)

In [ ]:
# Carrega o runtime nativo estrutural e mede memória/carga.
import psutil
import torch
from native_loader import load_native_packed_recovered_l78

process = psutil.Process(os.getpid())
HF_TOKEN = os.environ.get('HF_TOKEN')
CACHE_UNPACKED = os.environ.get('RPT_NATIVE_CACHE_UNPACKED', '0') == '1'

def format_bytes(n):
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    value = float(n)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f'{value:.2f} {unit}'
        value /= 1024

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def memory_snapshot(label):
    snap = {'label': label, 'rss_bytes': process.memory_info().rss, 'rss_human': format_bytes(process.memory_info().rss)}
    if torch.cuda.is_available():
        snap.update({
            'cuda_device': torch.cuda.get_device_name(0),
            'cuda_allocated_bytes': torch.cuda.memory_allocated(),
            'cuda_reserved_bytes': torch.cuda.memory_reserved(),
            'cuda_max_allocated_bytes': torch.cuda.max_memory_allocated(),
            'cuda_allocated_human': format_bytes(torch.cuda.memory_allocated()),
            'cuda_reserved_human': format_bytes(torch.cuda.memory_reserved()),
            'cuda_max_allocated_human': format_bytes(torch.cuda.max_memory_allocated()),
        })
    return snap

cleanup_memory()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

before_load = memory_snapshot('before_native_load')
started = time.perf_counter()
model, tokenizer, native_report = load_native_packed_recovered_l78(
    DIST_DIR,
    token=HF_TOKEN,
    device_map='auto',
    dtype='auto',
    cache_unpacked=CACHE_UNPACKED,
)
load_elapsed = time.perf_counter() - started
after_load = memory_snapshot('after_native_load')

print(json.dumps({'load_elapsed_sec': load_elapsed, 'before': before_load, 'after': after_load, 'native_report': native_report}, ensure_ascii=False, indent=2))

In [ ]:
# Smoke test de geração e avaliação opcional das 100 perguntas.
PROMPT_TEMPLATE = 'Responda de forma curta e direta.\nPergunta: {question}\nResposta:'
DEFAULT_PROMPTS = [
    'Qual a capital da França?',
    'Quanto é 12 x 8?',
    'Quem desenvolveu a teoria da relatividade?',
    'Em Python, qual palavra-chave define uma função?',
    'Se hoje é segunda-feira, que dia será depois de 3 dias?',
]

def generate_answer(question, max_new_tokens=32):
    prompt = PROMPT_TEMPLATE.format(question=question)
    inputs = tokenizer(prompt, return_tensors='pt')
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    started = time.perf_counter()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - started
    prompt_tokens = int(inputs['input_ids'].shape[-1])
    generated_tokens = int(output.shape[-1]) - prompt_tokens
    answer = tokenizer.decode(output[0][prompt_tokens:], skip_special_tokens=True).strip()
    return {
        'question': question,
        'answer': answer,
        'prompt_tokens': prompt_tokens,
        'generated_tokens': generated_tokens,
        'elapsed_sec': elapsed,
        'tokens_per_sec': generated_tokens / elapsed if elapsed > 0 else None,
    }

smoke_rows = [generate_answer(q) for q in DEFAULT_PROMPTS]
for row in smoke_rows:
    print(json.dumps(row, ensure_ascii=False))

RUN_FULL_100 = os.environ.get('RPT_NATIVE_RUN_FULL_100', '0') == '1'
eval_summary = None
eval_rows = []
if RUN_FULL_100:
    questions = json.loads(DATA_PATH.read_text(encoding='utf-8'))
    correct = 0
    for idx, item in enumerate(questions, 1):
        row = generate_answer(item['question'], max_new_tokens=24)
        expected = str(item['answer']).strip().lower()
        got = row['answer'].strip().lower()
        ok = expected in got or got in expected
        correct += int(ok)
        row.update({'expected': item['answer'], 'ok': ok, 'category': item.get('category')})
        eval_rows.append(row)
        print(f"{idx:03d}/{len(questions)} {'OK' if ok else 'FAIL'} | {item['question']} => {row['answer']}")
    eval_summary = {'total_questions': len(questions), 'correct': correct, 'accuracy': correct / len(questions)}
else:
    print('Avaliação 100 perguntas pulada. Defina RPT_NATIVE_RUN_FULL_100=1 para rodar.')

In [ ]:
# Salva relatório do runtime nativo estrutural.
result = {
    'notebook': '22_native_ternary_mlp_runtime',
    'purpose': 'Structural native ternary MLP runtime prototype for packed recovered L78.',
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'runtime_kind': 'native_structural_prototype',
    'is_native_ternary_runtime': True,
    'is_optimized_kernel': False,
    'cache_unpacked': CACHE_UNPACKED,
    'load_elapsed_sec': load_elapsed,
    'memory_before_load': before_load,
    'memory_after_load': after_load,
    'native_report': native_report,
    'smoke_rows': smoke_rows,
    'smoke_summary': {
        'num_prompts': len(smoke_rows),
        'total_generated_tokens': sum(r['generated_tokens'] for r in smoke_rows),
        'total_elapsed_sec': sum(r['elapsed_sec'] for r in smoke_rows),
        'avg_tokens_per_sec': sum(r['tokens_per_sec'] for r in smoke_rows if r['tokens_per_sec'] is not None) / max(1, sum(1 for r in smoke_rows if r['tokens_per_sec'] is not None)),
    },
    'eval_summary': eval_summary,
    'eval_rows': eval_rows,
    'baseline_notebook_20': {
        'materialized_hybrid_peak_vram': '5.33 GB',
        'materialized_hybrid_avg_tokens_per_sec_smoke': 7.746480522791005,
        'materialized_hybrid_load_elapsed_approx_sec': 152.38,
    },
    'next_step': 'If functional quality is preserved, implement optimized CUDA/Triton ternary matmul to avoid forward-time full unpack.',
}

out_path = RESULTS_DIR / f"22_native_ternary_mlp_runtime_{time.strftime('%Y%m%d_%H%M%S')}.json"
out_path.write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('saved:', out_path)
print(json.dumps(result['smoke_summary'], ensure_ascii=False, indent=2))
if eval_summary:
    print(json.dumps(eval_summary, ensure_ascii=False, indent=2))

## Critério de avanço

Avançar para kernel otimizado somente se:

1. `replaced_count == 78`;
2. o smoke test gera respostas coerentes;
3. a avaliação opcional de 100 perguntas não degrada de forma relevante;
4. a memória persistente após load cai ou pelo menos prova que os MLPs densos foram substituídos estruturalmente.

Se o protótipo estrutural funcionar, o próximo notebook deve ser `23_triton_ternary_mlp_kernel.ipynb`.